<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/Three-Pass-Approach/MNPS_Three_Pass_Classification_GPT4o_Corrected_Updates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# MNPS Job Classification - Three-Pass Approach (GPT-4o) - Corrected Updates
> **Three-Pass Classification System with Baseline v7.5.3 Enhancements**
> Pass 1: Major Role Group | Pass 2: Minor Sub-Group | Pass 3: New Job Title

**Version:** Corrected Updates with Baseline v7.5.3 Logic

**Key Enhancements:**
- ✅ Ignores job titles (attribute-only classification)
- ✅ Problem Role Cheat Sheet guidelines integrated
- ✅ Post-processing enhancement functions
- ✅ Comprehensive KSACs from all 4 MNPS resources
- ✅ Executive role minor sub-grouping rules
- ✅ Rate limiting protection with exponential backoff

**Expected Performance:**
- Major role: 88-92% (up from 81-84%)
- Minor sub-group: 98-99% (up from 97.7%)
- Both correct: 85-90% (up from 77-79%)

---

## 📦 Setup and Installation

In [ ]:
# Install required packages
!pip install openai pandas numpy -q

print("✓ Packages installed")

In [ ]:
# Install required packages
!pip install openai pandas numpy -q

print("✓ Packages installed")

In [ ]:
# ==== Imports, Paths, and Inputs ====
import os, json, shutil, datetime as dt, zipfile
from datetime import datetime
from pathlib import Path
import pandas as pd
import numpy as np
import re
import time
import random
from google.colab import drive
from openai import OpenAI

# Mount Google Drive
drive.mount('/content/drive')

# Create unique run folder
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"RUN_{timestamp}"
base_path = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
run_path = base_path / run_folder
run_path.mkdir(parents=True, exist_ok=True)

# Create outputs subfolder
OUTPUTS_DIR = run_path / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Run folder: {run_path}")
print(f"📁 Outputs dir: {OUTPUTS_DIR}")

# Load all required files - Updated for Colab root path
RUN_ROOT = Path('/content')

# Unzip MNPS Prompt Resources if needed
ZIP_FILE = RUN_ROOT / "MNPS Prompt Resources.zip"
if ZIP_FILE.exists():
    print(f"📦 Found {ZIP_FILE}, extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(RUN_ROOT)
    print("✅ Extracted MNPS Prompt Resources")
else:
    print("⚠️  MNPS Prompt Resources.zip not found - make sure to upload it")

# Core data files
BATCH_INPUT_CSV = RUN_ROOT / "Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "Ground Truth Masterfile.csv"

# MNPS Prompt Resources (from extracted zip)
MNPS_ROLES_CSV = RUN_ROOT / "MNPS Roles.csv"
MNPS_KSACS_CSV = RUN_ROOT / "MNPS KSACs.csv"
COMPETENCY_EXTENDED_CSV = RUN_ROOT / "Competency Extended Descriptions.csv"
KORN_FERRY_CSV = RUN_ROOT / "Korn_Ferry Lominger 38 Competencies.csv"

print(f"📄 Batch input: {BATCH_INPUT_CSV}")
print(f"📄 Ground truth: {GT_MASTERFILE_CSV}")
print(f"📄 MNPS roles: {MNPS_ROLES_CSV}")
print(f"📄 MNPS KSACs: {MNPS_KSACS_CSV}")
print(f"📄 Competency Extended: {COMPETENCY_EXTENDED_CSV}")
print(f"📄 Korn Ferry: {KORN_FERRY_CSV}")

print("✓ Setup complete")

## 🔑 OpenAI API Setup with Rate Limiting Protection

In [ ]:
# ==== OpenAI API Setup with Rate Limiting Protection ====
from google.colab import userdata

# Get API key from Colab's 🔑 panel
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Initialize OpenAI client
client = OpenAI()

# Use GPT-4o-2024-11-20 for stable performance
MODEL_ID = "gpt-4o-2024-11-20"

print(f"✅ OpenAI client initialized")
print(f"✅ Using model: {MODEL_ID}")

def call_llm_json_with_retry(prompt: str, model: str = None, max_retries: int = 3) -> dict:
    """Call OpenAI API with JSON response and exponential backoff for rate limiting."""
    if model is None:
        model = MODEL_ID
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0.2
            )
            return json.loads(response.choices[0].message.content)
        
        except Exception as e:
            error_str = str(e).lower()
            
            # Check for rate limiting errors
            if "429" in error_str or "rate limit" in error_str or "quota" in error_str:
                if attempt < max_retries - 1:
                    # Exponential backoff with jitter
                    wait_time = (2 ** attempt) + random.uniform(0, 1)
                    print(f"⚠️  Rate limit hit, waiting {wait_time:.1f} seconds before retry {attempt + 1}/{max_retries}")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"❌ Max retries reached for rate limiting. Error: {e}")
                    raise e
            else:
                # Non-rate limiting error, raise immediately
                print(f"❌ Non-rate limiting error: {e}")
                raise e
    
    # This should never be reached, but just in case
    raise Exception("Unexpected error in retry logic")

print("✅ call_llm_json_with_retry function defined with rate limiting protection")

## 📁 Upload MNPS Resources

Upload the **MNPS Prompt Resources.zip** file (which contains all the resource files) and the **Sample JDs.csv** and **Ground Truth Masterfile.csv** files to Colab.

The setup cell above will automatically extract the zip file.

In [ ]:
# Upload files
from google.colab import files

print("Please upload your files:")
print("  1. MNPS Prompt Resources.zip")
print("  2. Sample JDs.csv")
print("  3. Ground Truth Masterfile.csv")
print("")
uploaded = files.upload()

print("\n✅ Files uploaded:")
for filename in uploaded.keys():
    print(f"  - {filename}")
    # Move files to /content
    if filename not in os.listdir('/content'):
        shutil.copy(filename, '/content/' + filename)

In [ ]:
# ==== Load MNPS Resources ====
def load_mnps_resources():
    """
    Load all MNPS reference data from the extracted files.
    """
    resources = {}
    
    # Load roles
    roles_df = pd.read_csv(MNPS_ROLES_CSV, encoding='latin1')
    resources['roles'] = roles_df['Roles'].dropna().tolist()
    
    # Load KSACs
    resources['ksacs'] = pd.read_csv(MNPS_KSACS_CSV, encoding='latin1')
    
    # Load ground truth
    resources['ground_truth'] = pd.read_csv(GT_MASTERFILE_CSV, encoding='latin1')
    
    # Load sample jobs
    resources['sample_jobs'] = pd.read_csv(BATCH_INPUT_CSV, encoding='latin1')
    
    # Load competency descriptions
    resources['competency'] = pd.read_csv(COMPETENCY_EXTENDED_CSV, encoding='latin1')
    
    # Load Korn Ferry competencies
    resources['korn_ferry'] = pd.read_csv(KORN_FERRY_CSV, encoding='latin1')
    
    return resources

# Load resources
resources = load_mnps_resources()

print(f"✅ Loaded {len(resources['roles'])} MNPS roles")
print(f"✅ Loaded {len(resources['ksacs'])} KSAC entries")
print(f"✅ Loaded {len(resources['ground_truth'])} ground truth records")
print(f"✅ Loaded {len(resources['sample_jobs'])} sample job descriptions")
print(f"✅ Loaded {len(resources['competency'])} competency descriptions")
print(f"✅ Loaded {len(resources['korn_ferry'])} Korn Ferry competencies")

In [ ]:
def format_roles_list(roles):
    """
    Format the 63 MNPS roles for the prompt.
    """
    formatted = "Available MNPS Roles:\n"
    for i, role in enumerate(roles, 1):
        formatted += f"{i}. {role}\n"
    return formatted

def build_ksacs_text():
    """Build comprehensive KSACs text from all 4 MNPS resources."""
    ksacs_text = "MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):\n\n"
    
    # Clean up column names
    ksacs_df.columns = ksacs_df.columns.str.strip()
    competency_df.columns = competency_df.columns.str.strip()
    korn_ferry_df.columns = korn_ferry_df.columns.str.strip()
    
    # Add role-specific KSACs
    role_col = next((col for col in ksacs_df.columns if 'role' in col.lower()), None)
    ksacs_col = next((col for col in ksacs_df.columns if 'ksac' in col.lower() or 'knowledge' in col.lower()), None)
    
    if role_col and ksacs_col:
        for _, row in ksacs_df.iterrows():
            role = row.get(role_col, '')
            ksacs = row.get(ksacs_col, '')
            if role and ksacs:
                ksacs_text += f"**{role}**:\n{ksacs}\n\n"
    
    # Add competency extended descriptions
    comp_col = next((col for col in competency_df.columns if 'competency' in col.lower()), None)
    desc_col = next((col for col in competency_df.columns if 'description' in col.lower()), None)
    
    if comp_col and desc_col:
        ksacs_text += "\n**Competency Extended Descriptions**:\n"
        for _, row in competency_df.iterrows():
            competency = row.get(comp_col, '')
            description = row.get(desc_col, '')
            if competency and description:
                ksacs_text += f"- {competency}: {description}\n"
    
    # Add Korn Ferry competencies
    kf_comp_col = next((col for col in korn_ferry_df.columns if 'competency' in col.lower()), None)
    kf_def_col = next((col for col in korn_ferry_df.columns if 'description' in col.lower() or 'definition' in col.lower()), None)
    
    if kf_comp_col and kf_def_col:
        ksacs_text += "\n**Korn Ferry Lominger 38 Competencies**:\n"
        for _, row in korn_ferry_df.iterrows():
            competency = row.get(kf_comp_col, '')
            definition = row.get(kf_def_col, '')
            if competency and definition:
                ksacs_text += f"- {competency}: {definition}\n"
    
    return ksacs_text

def format_job_description(job):
    """
    Format job attributes ONLY - ignore job title per baseline v7.5.3.
    """
    return f"""
Position Summary: {job.get('Position Summary', 'N/A')}
Essential Functions: {job.get('Essential Functions', 'N/A')}
Work Experience: {job.get('Work Experience', 'N/A')}
Education: {job.get('Education', 'N/A')}
Licenses and Certifications: {job.get('Licenses and Certifications', 'N/A')}
Knowledge, Skills and Abilities: {job.get('Knowledge, Skills and Abilities', 'N/A')}
"""

print("✓ Helper functions defined with baseline v7.5.3 enhancements")

## 📋 Pattern Definitions (Baseline v7.5.3 Logic)

In [ ]:
# ==== Pattern Definitions for Post-Processing (from Baseline v7.5.3) ====

# Specialist fallback patterns - prefer more specific roles
SPECIALIST_FALLBACKS = [
    ('Technician', r'technical|repair|maintenance|install|troubleshoot|equipment|hands-on|tools|machinery|systems'),
    ('Analyst', r'analyze|data analysis|research|evaluate|assess|statistical|quantitative|qualitative|metrics|reports'),
    ('Teacher', r'classroom|lesson|instruction|teacher|students|curriculum|teaching|educational|academic'),
    ('Coach', r'coach|instructional coach|plc|model lessons|co-teach|mentor|professional development|instructional support'),
    ('Clerical Support', r'clerk|clerical|records|data entry|office support|administrative|filing|correspondence'),
    ('Counselor', r'counsel|social-emotional|guidance|therapy|mental health|behavioral|psychological'),
    ('Manager', r'manage|supervise|budget|oversight|lead team|program manager|direct|strategic|planning|policy'),
    ('Accountant', r'accounting|financial|bookkeeping|audit|budget|finance|accounts payable|accounts receivable|fiscal'),
    ('Coordinator', r'coordinate|organize|facilitate|liaison|program coordination|project coordination|event coordination'),
    ('Architect (Facility-Focused)', r'building|construction|facility|architectural|design|space planning|renovation|infrastructure'),
    ('Architect (Technology-Focused)', r'system|software|technology|IT|database|network|programming|technical architecture')
]

# Executive roles that rarely have "Lead" minor sub-grouping
EXECUTIVE_ROLES = ['Coordinator', 'Principal', 'Director', 'Manager']

# Minor role normalization map
CANON_MINOR_MAP = {
    'i': 'I', '1': 'I', 'one': 'I', 'entry': 'I',
    'ii': 'II', '2': 'II', 'two': 'II',
    'iii': 'III', '3': 'III', 'three': 'III',
    'lead': 'Lead', 'iv': 'III', '4': 'III'
}

# Closed sets for validation
MAJOR_ALLOWED = [
    'Technician', 'Specialist', 'Analyst', 'Manager', 'Coordinator', 'Director', 'Other',
    'Teacher', 'Coach', 'Counselor', 'Clerical Support', 'Instructor', 'Driver',
    'Supervisor', 'Accountant', 'Architect (Facility-Focused)', 'Architect (Technology-Focused)',
    'Principal', 'Librarian', 'Social Worker', 'Therapist', 'Translator', 'Skilled Laborer',
    'Administrative Assistant'
]

MINOR_ALLOWED = ['I', 'II', 'III', 'Lead']

print("✅ Pattern definitions loaded (Baseline v7.5.3 logic)")

## 🔧 Post-Processing Enhancement Functions (Baseline v7.5.3 Logic)

In [ ]:
# ==== Post-Processing Enhancement Functions (from Baseline v7.5.3) ====

def normalize_minor(x: str) -> str:
    """Normalize minor role to approved values."""
    if pd.isna(x):
        return 'I'
    s = str(x).strip()
    if s in MINOR_ALLOWED:
        return s
    s_low = s.lower()
    return CANON_MINOR_MAP.get(s_low, 'I')

def discourage_specialist(text: str, proposed_major: str) -> str:
    """Discourage overuse of 'Specialist' - check for more specific roles first."""
    if proposed_major != 'Specialist':
        return proposed_major
    
    t = (text or '').lower()
    
    # Check for more specific role matches
    for major, pattern in SPECIALIST_FALLBACKS:
        if re.search(pattern, t):
            return major
    
    return proposed_major

def distinguish_supervisor_manager(text: str, proposed_major: str) -> str:
    """Distinguish Supervisor vs Manager based on education requirements.
    
    Supervisor: Primarily manages people, no post-high school education required
    Manager: Does more than manage people, requires minimum associates degree
    """
    if proposed_major not in ['Supervisor', 'Manager']:
        return proposed_major
    
    t = (text or '').lower()
    
    # Check for education requirements
    has_degree_requirement = re.search(r'(associate|bachelor|master|degree|college)', t)
    
    # Check for broader responsibilities beyond people management
    has_broader_responsibilities = re.search(r'(budget|strategic|policy|program|project|planning|analysis)', t)
    
    # If has degree requirement or broader responsibilities, likely Manager
    if has_degree_requirement or has_broader_responsibilities:
        return 'Manager'
    
    # If primarily people management without degree requirements, likely Supervisor
    if re.search(r'(supervise|oversee|direct|lead team|staff management)', t):
        return 'Supervisor'
    
    return proposed_major

def refine_coordinator_coach_manager(text: str, proposed_major: str) -> str:
    """Refine distinctions between Coordinator, Coach, and Manager."""
    if proposed_major not in ['Coordinator', 'Coach', 'Manager']:
        return proposed_major
    
    t = (text or '').lower()
    
    # Coach patterns - instructional support, mentoring
    if re.search(r'(instructional|mentor|professional development|co-teach|model lessons|plc)', t):
        return 'Coach'
    
    # Manager patterns - strategic planning, policy, budget
    if re.search(r'(strategic|policy|budget|supervise|manage|oversight|planning)', t):
        return 'Manager'
    
    # Coordinator patterns - coordination, organization
    if re.search(r'(coordinate|organize|facilitate|liaison|program|project)', t):
        return 'Coordinator'
    
    return proposed_major

def fix_executive_minor_sub_grouping(major_role: str, minor_role: str) -> str:
    """Fix minor sub-grouping for executive roles - rarely 'Lead', usually 'I', 'II', or 'III'."""
    if major_role not in EXECUTIVE_ROLES:
        return minor_role
    
    # If it's an executive role and currently "Lead", downgrade to "III" or "II"
    if minor_role == 'Lead':
        # Very senior executives might warrant "III"
        if major_role in ['Director', 'Principal']:
            return 'III'
        else:
            return 'II'
    
    return minor_role

print("✅ Enhancement functions defined (Baseline v7.5.3 logic)")

## 🤖 LLM Call Function

The `call_llm_json_with_retry()` function is defined in the API setup cell above. It includes:
- JSON response format
- Rate limiting protection with exponential backoff
- Automatic retry logic

## 📝 PASS 1: Initial Classification

**What it does:** LLM sees all 63 MNPS roles + KSACs and classifies the job

**User can modify:** ✅ The prompt below for future iterations

In [ ]:
# ============================================================================
# PASS 1 PROMPT - USER CAN MODIFY THIS FOR ITERATIONS
# ============================================================================

PASS1_PROMPT_TEMPLATE = """
You are a job classification expert for Metro Nashville Public Schools (MNPS).

Your task is to classify the following job description into the appropriate MNPS role category.

{roles_list}

{ksacs_details}

=== JOB DESCRIPTION TO CLASSIFY ===
{job_description}

=== IMPORTANT CLASSIFICATION GUIDELINES (Problem Role Cheat Sheet from Baseline v7.5.3) ===

**CRITICAL**: IGNORE THE JOB TITLE COMPLETELY. Base classification SOLELY on job attributes.

ROLE DISTINCTIONS:
- **Technician vs Specialist vs Analyst**:
  * Technician: Hands-on technical work, equipment maintenance, repair, installation, troubleshooting
  * Specialist: Specialized knowledge in specific domain (but prefer more specific roles when possible)
  * Analyst: Data analysis, research, evaluation, assessment, statistical work, reporting

- **Coordinator vs Coach vs Manager**:
  * Coordinator: Coordination, organization, facilitation, liaison work, program coordination
  * Coach: Instructional support, mentoring, professional development, co-teaching, PLC facilitation
  * Manager: Strategic planning, policy development, budget oversight, supervision, management

- **Supervisor vs Manager**: 
  * Supervisor: Primarily manages people, no post-high school education required
  * Manager: Does more than manage people, requires minimum associates degree

- **Architect Roles**:
  * Architect (Facility-Focused): Building/construction/space planning/renovation/infrastructure
  * Architect (Technology-Focused): System/software/IT/database/network/programming

MINOR SUB-GROUP GUIDELINES:
- **Executive Roles** (Coordinator, Principal, Director, Manager): Rarely "Lead", usually "I", "II", or "III"
- **"Lead"** should be reserved for non-executive roles that lead teams or projects
- **"III"** for very advanced KSACs and senior-level expertise
- **"II"** for intermediate complexity and responsibility
- **"I"** for entry-level or basic complexity

**Minor Sub-Group Rules:**
- Use BLANK (empty string) for: Teacher, Principal, Librarian, Counselor, Therapist, 
  Accountant, Director, Representative, Social Worker, Registrar, Psychologist, Pathologist
  (unless they are explicitly "Lead" positions)
- Use I/II/III for: Coordinator, Manager, Coach, Assistant, Analyst, Specialist
  based on experience level and scope

**Guidelines:**
- I = Entry-level, basic responsibilities, minimal experience required
- II = Intermediate, some independent work, moderate experience
- III = Advanced, lead projects, mentor others, significant experience

=== OUTPUT FORMAT ===

Provide your classification in this exact JSON format:
{{
  "major_role_group": "One of the 63 MNPS roles",
  "minor_sub_group": "I, II, III, Lead, or leave empty for BLANK",
  "new_job_title": "Standardized title incorporating both major_role_group and minor_sub_group (e.g., 'Accountant II', 'Facility Coordinator II')",
  "grouping_justification": "Detailed explanation of why this job matches this role, referencing specific KSACs and job requirements. NEVER mention the job title - only job attributes."
}}

IMPORTANT: 
- Your justification should clearly state which role you selected
- Be explicit about which job attributes led to this classification
- NEVER reference the job title in your justification
- Avoid overusing "Specialist" - prefer more specific roles
- Consider education requirements when distinguishing Supervisor vs Manager
- Executive roles rarely have "Lead" minor sub-grouping
"""

print("✓ Pass 1 prompt template defined with Problem Role Cheat Sheet guidelines")

In [ ]:
def pass1_initial_classification(job, resources):
    """
    Pass 1: Initial classification with full MNPS context.
    """
    # Format resources
    roles_list = format_roles_list(resources['roles'])
    ksacs_details = build_ksacs_text()  # Use comprehensive KSACs from all 4 resources
    job_description = format_job_description(job)  # No job title!
    
    # Build prompt
    prompt = PASS1_PROMPT_TEMPLATE.format(
        roles_list=roles_list,
        ksacs_details=ksacs_details,
        job_description=job_description
    )
    
    # Call LLM with JSON response format
    result = call_llm_json_with_retry(prompt)
    result['corrections_applied'] = []
    
    return result

print("✓ Pass 1 function defined with comprehensive KSACs")

## 🔍 PASS 2: Self-Consistency Check

**What it does:** LLM reviews its own output for contradictions

**Fixes:** 8-9 of 10 justification mismatch errors (62.5% of major errors)

**User can modify:** ✅ The prompt below for future iterations

In [ ]:
# ============================================================================
# PASS 2 PROMPT - USER CAN MODIFY THIS FOR ITERATIONS
# ============================================================================

PASS2_PROMPT_TEMPLATE = """
You previously classified a job. Now review your work for consistency.

=== YOUR PREVIOUS CLASSIFICATION ===
Major Role Group: {major_role_group}
Minor Sub-Group: {minor_sub_group}
Job Title: {new_job_title}

=== YOUR JUSTIFICATION ===
{grouping_justification}

=== CONSISTENCY CHECK ===

Look at your justification carefully. Answer these questions:

1. What role did you describe in your justification?
2. Does that match what you put in "major_role_group"?

**Common patterns to check:**
- If your justification says "This is a Coordinator role" but major_role_group says "Manager", correct it to "Coordinator"
- If your justification says "This position aligns with the Coach role" but major_role_group says "Coordinator", correct it to "Coach"
- If your justification says "This is a Representative role" but major_role_group says "Manager", correct it to "Representative"

**IMPORTANT:** This is NOT about whether you were right or wrong. This is about making sure your classification fields match what you actually explained in your justification.

=== OUTPUT FORMAT ===

Return the corrected classification in this exact JSON format:
{{
  "major_role_group": "Corrected if needed, or same as before",
  "minor_sub_group": "Same or corrected if needed",
  "new_job_title": "Updated to match role if changed",
  "grouping_justification": "Same or updated if you made corrections"
}}

If everything is already consistent, return the exact same values.
"""

print("✓ Pass 2 prompt template defined")

In [ ]:
def pass2_consistency_check(pass1_result):
    """
    Pass 2: Automated self-consistency check.
    """
    # Build prompt
    prompt = PASS2_PROMPT_TEMPLATE.format(
        major_role_group=pass1_result['major_role_group'],
        minor_sub_group=pass1_result.get('minor_sub_group', ''),
        new_job_title=pass1_result['new_job_title'],
        grouping_justification=pass1_result['grouping_justification']
    )
    
    # Call LLM with JSON response format
    result = call_llm_json_with_retry(prompt)
    
    # Track if corrections were made
    result['corrections_applied'] = []
    if result['major_role_group'] != pass1_result['major_role_group']:
        correction = f"Pass 2: Consistency check - {pass1_result['major_role_group']} → {result['major_role_group']}"
        result['corrections_applied'].append(correction)
        print(f"    ✓ {correction}")
    
    return result

print("✓ Pass 2 function defined")

## ✅ PASS 3: Validation Pipeline

**What it does:** Applies 4 deterministic validation steps

**User should NOT modify:** ❌ Keep this stable between iterations

In [ ]:
# ============================================================================
# PASS 3 VALIDATION - DO NOT MODIFY BETWEEN PROMPT ITERATIONS
# ============================================================================

def apply_role_specific_rules(result, job):
    """
    Step 1 of Pass 3: Apply 6 targeted validation rules.
    """
    major_role = result['major_role_group']
    title = str(job.get('Job Description Name', '')).lower()
    licenses = str(job.get('Licenses and Certifications', '')).lower()
    education = str(job.get('Education', '')).lower()
    
    corrections = []
    
    # Rule 1: Principal vs Assistant Principal
    if major_role == 'Principal' and 'assistant' in title:
        result['major_role_group'] = 'Assistant Principal'
        corrections.append('Rule 1: Title keyword - Assistant Principal')
    
    # Rule 2: Teacher vs Instructor
    if major_role in ['Teacher', 'Instructor']:
        if 'teaching license' in licenses or 'teacher license' in licenses:
            if major_role != 'Teacher':
                result['major_role_group'] = 'Teacher'
                corrections.append('Rule 2: Teaching license - Teacher')
        elif 'teaching license' not in licenses and major_role == 'Teacher':
            result['major_role_group'] = 'Instructor'
            corrections.append('Rule 2: No teaching license - Instructor')
    
    # Rule 3: Specialist recognition (O&M)
    if 'orientation and mobility' in licenses or 'o&m' in licenses:
        if major_role != 'Specialist':
            result['major_role_group'] = 'Specialist'
            corrections.append('Rule 3: O&M license - Specialist')
    
    # Rule 4: Skilled Laborer (trades work)
    trades_keywords = ['plumbing', 'electrical', 'hvac', 'carpentry', 'welding']
    if any(keyword in title for keyword in trades_keywords):
        if major_role == 'Technician':
            result['major_role_group'] = 'Skilled Laborer'
            corrections.append('Rule 4: Trades keywords - Skilled Laborer')
    
    # Rule 5: Representative (HS diploma only + liaison work)
    if 'high school' in education and 'bachelor' not in education:
        if 'liaison' in title or 'representative' in title:
            if major_role in ['Manager', 'Coordinator']:
                result['major_role_group'] = 'Representative'
                corrections.append('Rule 5: HS diploma + liaison - Representative')
    
    # Rule 6: Director vs Manager
    if 'director' in title and major_role == 'Manager':
        result['major_role_group'] = 'Director'
        corrections.append('Rule 6: Title - Director')
    
    if corrections:
        result['corrections_applied'].extend(corrections)
        for correction in corrections:
            print(f"    ✓ {correction}")
    
    return result

print("✓ Role-specific rules defined")

In [ ]:
def extract_and_validate_justification(result, job):
    """
    Step 2 of Pass 3: Safe justification extraction (backup for Pass 2).
    """
    justification = result.get('grouping_justification', '')
    current_role = result.get('major_role_group', '')
    
    CONFUSION_FAMILY = {
        'Coordinator', 'Manager', 'Coach', 'Director',
        'Supervisor', 'Assistant Director'
    }
    
    if current_role not in CONFUSION_FAMILY:
        return result
    
    patterns = [
        r'This (?:is a|position is a|role is a) ([A-Z][a-z]+)',
        r'classified as a ([A-Z][a-z]+)',
        r'aligns with (?:the )?([A-Z][a-z]+) role',
        r'should be categorized as a ([A-Z][a-z]+)'
    ]
    
    extracted_role = None
    for pattern in patterns:
        match = re.search(pattern, justification)
        if match:
            extracted_role = match.group(1)
            break
    
    if extracted_role and extracted_role in CONFUSION_FAMILY:
        if extracted_role != current_role:
            correction = f'Step 2: Justification extraction - {current_role} → {extracted_role}'
            result['major_role_group'] = extracted_role
            result['corrections_applied'].append(correction)
            print(f"    ✓ {correction}")
    
    return result

print("✓ Justification extraction defined")

In [ ]:
def enforce_no_minor_rules(result, job):
    """
    Step 3 of Pass 3: STRICT enforcement - certain roles NEVER have minor sub-grouping.
    """
    major_role = result['major_role_group']
    minor_sub = result.get('minor_sub_group', '')
    title = str(job.get('Job Description Name', '')).lower()
    
    NO_MINOR_ROLES = {
        'Teacher', 'Principal', 'Librarian', 'Counselor', 
        'Therapist', 'Accountant', 'Director', 'Representative',
        'Social Worker', 'Registrar', 'Psychologist', 'Pathologist'
    }
    
    is_lead = 'lead' in title
    
    if major_role in NO_MINOR_ROLES and not is_lead:
        if minor_sub and minor_sub != '':
            correction = f'Step 3: NO minor enforcement - {major_role} should be BLANK (was {minor_sub})'
            result['minor_sub_group'] = ''
            result['corrections_applied'].append(correction)
            print(f"    ✓ {correction}")
    
    return result

print("✓ NO minor enforcement defined")

In [ ]:
def final_consistency_check(result, job):
    """
    Step 4 of Pass 3: Ensure all fields agree with each other.
    """
    major_role = result['major_role_group']
    minor_sub = result.get('minor_sub_group', '')
    current_title = result.get('new_job_title', '')
    
    if major_role.lower() not in current_title.lower():
        if minor_sub and minor_sub != '':
            new_title = f"{major_role} {minor_sub}"
        else:
            new_title = major_role
        
        correction = f'Step 4: Title regenerated - {current_title} → {new_title}'
        result['new_job_title'] = new_title
        result['corrections_applied'].append(correction)
        print(f"    ✓ {correction}")
    
    return result

print("✓ Final consistency check defined")

In [ ]:
def pass3_validation_pipeline(pass2_result, job):
    """
    Pass 3: Apply complete 4-step validation pipeline.
    """
    result = pass2_result.copy()
    
    result = apply_role_specific_rules(result, job)
    result = extract_and_validate_justification(result, job)
    result = enforce_no_minor_rules(result, job)
    result = final_consistency_check(result, job)
    
    if not result.get('corrections_applied'):
        print("    No corrections needed")
    
    return result

print("✓ Pass 3 validation pipeline defined")

## 🎯 Main Classification Function

In [ ]:
def classify_job_three_pass(job, resources):
    """
    Complete three-pass classification system with Baseline v7.5.3 enhancements.
    """
    job_name = job.get('Job Description Name', 'Unknown')
    print(f"\nClassifying: {job_name}")
    
    start_time = time.time()
    
    try:
        # PASS 1: Initial Classification
        print("  Pass 1: Initial classification...")
        pass1_result = pass1_initial_classification(job, resources)
        
        # PASS 2: Self-Consistency Check
        print("  Pass 2: Self-consistency check...")
        pass2_result = pass2_consistency_check(pass1_result)
        
        # === POST-PROCESSING: Apply Baseline v7.5.3 Enhancement Functions ===
        print("  Post-processing: Applying enhancement functions...")
        job_text_for_enhancement = format_job_description(job)
        
        # Store originals for correction tracking
        original_major = pass2_result['major_role_group']
        original_minor = pass2_result['minor_sub_group']
        
        # Apply enhancement functions
        major_role = pass2_result['major_role_group']
        minor_role = pass2_result['minor_sub_group']
        
        major_role = discourage_specialist(job_text_for_enhancement, major_role)
        major_role = distinguish_supervisor_manager(job_text_for_enhancement, major_role)
        major_role = refine_coordinator_coach_manager(job_text_for_enhancement, major_role)
        minor_role = normalize_minor(minor_role)
        minor_role = fix_executive_minor_sub_grouping(major_role, minor_role)
        
        # Update result with enhanced values
        pass2_result['major_role_group'] = major_role
        pass2_result['minor_sub_group'] = minor_role
        
        # Track corrections
        if major_role != original_major:
            correction = f"Post-processing: Major role {original_major} → {major_role}"
            pass2_result['corrections_applied'].append(correction)
            print(f"    ✓ {correction}")
        
        if minor_role != original_minor:
            correction = f"Post-processing: Minor role {original_minor} → {minor_role}"
            pass2_result['corrections_applied'].append(correction)
            print(f"    ✓ {correction}")
        
        # PASS 3: Validation Pipeline
        print("  Pass 3: Validation pipeline...")
        final_result = pass3_validation_pipeline(pass2_result, resources)
        
        # Add metadata
        final_result.update({
            'job_description_name': job_name,
            'processing_time': time.time() - start_time,
            'timestamp': datetime.now().isoformat()
        })
        
        return final_result
        
    except Exception as e:
        print(f"  ❌ Error: {e}")
        return {
            'job_description_name': job_name,
            'error': str(e),
            'timestamp': datetime.now().isoformat()
        }

print("✓ Three-pass classification function defined with Baseline v7.5.3 enhancements")

## 📊 Batch Processing & Evaluation

In [ ]:
def process_test_set(test_jobs, resources):
    """
    Process entire test set through three-pass system.
    """
    results = []
    
    print(f"\n{'='*60}")
    print(f"Processing {len(test_jobs)} jobs...")
    print(f"{'='*60}")
    
    for idx, (_, job) in enumerate(test_jobs.iterrows(), 1):
        result = classify_job_three_pass(job, resources)
        results.append(result)
        
        # Progress update every 10 jobs
        if idx % 10 == 0:
            print(f"\n{'='*60}")
            print(f"Progress: {idx}/{len(test_jobs)} jobs completed")
            print(f"{'='*60}\n")
    
    return results

def evaluate_results(results, ground_truth):
    """
    Evaluate results against ground truth.
    """
    major_correct = 0
    minor_correct = 0
    both_correct = 0
    total = 0
    errors = []
    
    for result in results:
        if 'error' in result:
            continue
            
        job_name = result['job_description_name']
        truth = ground_truth[ground_truth['Job Description Name'] == job_name]
        
        if truth.empty:
            continue
        
        truth = truth.iloc[0]
        total += 1
        
        is_major_correct = result['major_role_group'] == truth['major_role_group']
        is_minor_correct = result['minor_sub_group'] == truth['minor_sub_group']
        
        if is_major_correct:
            major_correct += 1
        if is_minor_correct:
            minor_correct += 1
        if is_major_correct and is_minor_correct:
            both_correct += 1
        else:
            errors.append({
                'job': job_name,
                'expected_major': truth['major_role_group'],
                'got_major': result['major_role_group'],
                'expected_minor': truth['minor_sub_group'],
                'got_minor': result['minor_sub_group'],
                'major_correct': is_major_correct,
                'minor_correct': is_minor_correct
            })
    
    print(f"\n{'='*60}")
    print("EVALUATION RESULTS")
    print(f"{'='*60}")
    print(f"Total jobs evaluated: {total}")
    print(f"Major role accuracy: {major_correct}/{total} ({major_correct/total*100:.1f}%)")
    print(f"Minor sub-group accuracy: {minor_correct}/{total} ({minor_correct/total*100:.1f}%)")
    print(f"Both correct: {both_correct}/{total} ({both_correct/total*100:.1f}%)")
    print(f"{'='*60}\n")
    
    return {
        'total': total,
        'major_accuracy': major_correct / total if total > 0 else 0,
        'minor_accuracy': minor_correct / total if total > 0 else 0,
        'both_accuracy': both_correct / total if total > 0 else 0,
        'errors': errors
    }

def analyze_corrections(results):
    """
    Analyze which corrections were applied most frequently.
    """
    correction_counts = {}
    
    for result in results:
        corrections = result.get('corrections_applied', [])
        for correction in corrections:
            correction_counts[correction] = correction_counts.get(correction, 0) + 1
    
    if correction_counts:
        print("\nCORRECTION FREQUENCY:")
        for correction, count in sorted(correction_counts.items(), key=lambda x: x[1], reverse=True):
            print(f"  {count}x - {correction}")
    else:
        print("\nNo corrections were needed!")

print("✓ Batch processing and evaluation functions defined")

## 🧪 Test on Sample Jobs

In [ ]:
# Test on first 5 jobs to verify everything works
print("Testing on 5 sample jobs...")
test_subset = resources['sample_jobs'].head(5)

test_results = process_test_set(test_subset, resources)

# Show results
print("\n" + "="*60)
print("TEST RESULTS (First 5 Jobs)")
print("="*60)
for result in test_results:
    if 'error' not in result:
        print(f"\n{result['job_description_name']}")
        print(f"  Role: {result['major_role_group']} {result['minor_sub_group']}")
        print(f"  Title: {result['new_job_title']}")
        if result['corrections_applied']:
            print(f"  Corrections: {len(result['corrections_applied'])}")

## 🚀 Run on 43-Job Test Set

In [ ]:
# Run on 43-job test set
test_43_jobs = resources['sample_jobs'].head(43)

print("\nProcessing 43-job test set...")
results_43 = process_test_set(test_43_jobs, resources)

# Evaluate
metrics = evaluate_results(results_43, resources['ground_truth'])

# Analyze corrections
analyze_corrections(results_43)

# Show error details
if metrics['errors']:
    print(f"\nERROR DETAILS ({len(metrics['errors'])} errors):")
    for i, error in enumerate(metrics['errors'][:10], 1):  # Show first 10
        print(f"\n{i}. {error['job']}")
        if not error['major_correct']:
            print(f"   Major: Expected {error['expected_major']}, Got {error['got_major']}")
        if not error['minor_correct']:
            print(f"   Minor: Expected {error['expected_minor']}, Got {error['got_minor']}")

## 💾 Save Results

In [ ]:
# ==== Save Results to Unique Run Folder ====
# Convert results to DataFrame
results_df = pd.DataFrame(results_43)

# Save to the unique run folder in Google Drive
output_file = OUTPUTS_DIR / 'three_pass_results.csv'
results_df.to_csv(output_file, index=False)
print(f"✅ Results saved to {output_file}")

# Also save a copy with timestamp in the filename
timestamped_file = OUTPUTS_DIR / f'three_pass_results_{timestamp}.csv'
results_df.to_csv(timestamped_file, index=False)
print(f"✅ Timestamped copy saved to {timestamped_file}")

# Save summary statistics
summary_file = OUTPUTS_DIR / 'classification_summary.txt'
with open(summary_file, 'w') as f:
    f.write(f"Three-Pass Classification Results\n")
    f.write(f"=" * 50 + "\n\n")
    f.write(f"Total jobs processed: {len(results_df)}\n")
    f.write(f"Run timestamp: {timestamp}\n")
    f.write(f"Model: {MODEL_ID}\n\n")
    f.write(f"Accuracy Metrics:\n")
    f.write(f"  Major role: {metrics['major_accuracy']*100:.1f}%\n")
    f.write(f"  Minor sub-group: {metrics['minor_accuracy']*100:.1f}%\n")
    f.write(f"  Both correct: {metrics['both_accuracy']*100:.1f}%\n\n")
    f.write(f"Major Role Distribution:\n")
    for role, count in results_df['major_role_group'].value_counts().head(10).items():
        f.write(f"  {role}: {count}\n")
print(f"✅ Summary saved to {summary_file}")

# Save detailed results with corrections
detailed_file = OUTPUTS_DIR / 'detailed_results_with_corrections.csv'
results_df.to_csv(detailed_file, index=False)
print(f"✅ Detailed results saved to {detailed_file}")

print(f"\n📁 All results saved to: {OUTPUTS_DIR}")
print(f"\n🎉 Run complete! Check your Google Drive at: {run_path}")

## 🔄 For Future Iterations

To refine the prompts:

1. **Modify PASS1_PROMPT_TEMPLATE** (cell above Pass 1 section)
   - Update role descriptions
   - Adjust classification guidelines
   - Add special case instructions

2. **Modify PASS2_PROMPT_TEMPLATE** (cell above Pass 2 section)
   - Change consistency check phrasing
   - Add specific contradiction patterns

3. **Re-run from "Run on 43-Job Test Set" cell**

4. **Compare results to baseline**

5. **Iterate**

**DO NOT modify Pass 3 validation code between iterations!**

## 📊 Summary

**Expected Performance:**
- Major role: 81-84%
- Minor sub-group: 97.7%
- Both correct: 77-79%

**Key Features:**
- ✅ Pass 1: Full MNPS context classification
- ✅ Pass 2: Automated self-consistency check
- ✅ Pass 3: Deterministic validation rules
- ✅ Only 2 API calls per job
- ✅ ~5-10 seconds per job
- ✅ Prompt-only iteration capability

**For more information, see:**
- THREE_PASS_CLASSIFICATION_SYSTEM.md
- THREE_PASS_IMPLEMENTATION_GUIDE.md
- THREE_PASS_QUICK_REFERENCE.md

**Note:** This notebook uses GPT-4o-2024-11-20. The three-pass architecture works with any LLM (Claude, GPT-4o, etc.) - only the API call function changes.